# Prepare Transit Catchment Diversity Data (US)

This notebook prepares all necessary data for computing transit catchment diversity:

1. **Download GTFS feeds** from Mobility Database catalog
2. **Merge GTFS feeds** per city for r5r
3. **Extract POI locations** for routing origins
4. **Download Census tract shapefiles** for centroids
5. **Set up r5r configuration**

The actual routing will be done in R using r5r (much faster than Python alternatives).

In [3]:
%cd /workspace

from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Dict, Optional

import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO

# Use relative paths (notebooks run in Docker at /workspace)
GTFS_DIR = Path('dbs/gtfs')
GTFS_DIR.mkdir(parents=True, exist_ok=True)

CITIES_DIR = Path('dbs/us_foot_traffic/cities')
ROUTING_DIR = Path('dbs/routing')
ROUTING_DIR.mkdir(parents=True, exist_ok=True)

print(f"GTFS directory: {GTFS_DIR}")
print(f"Routing directory: {ROUTING_DIR}")

/workspace
GTFS directory: dbs/gtfs
Routing directory: dbs/routing


## 1. GTFS Feed Configuration

Using the [Mobility Database](https://mobilitydatabase.org/) catalog to find and download GTFS feeds.

**Advantages over hardcoded URLs:**
- Always gets latest feed versions
- Handles authentication requirements
- Pattern matching catches regional providers (NJ Transit, MARC, VRE, etc.)
- Saves manifest for auditing

In [2]:
# =========================
# Configuration
# =========================
CATALOG_URL = "https://files.mobilitydatabase.org/feeds_v2.csv"

# Add API keys here if you need them for feeds with authentication_type 1 or 2.
# Example:
# API_KEYS = {
#     "api.wmata.com": "YOUR_WMATA_KEY",
# }
API_KEYS: Dict[str, str] = {}

# Manual provider matching rules.
# These are intentionally explicit, because filtering only by municipality can miss
# important regional providers like NJ Transit, MARC, VRE, etc.
CITY_RULES = {
    "new_york": {
        "country_code": "US",
        "subdivision_names": ["New York", "New Jersey", "Connecticut"],
        "provider_patterns": [
            r"\bMTA\b",
            r"New York City Transit",
            r"MTA Bus",
            r"Long Island Rail",
            r"Metro-North",
            r"\bPATH\b",
            r"NJ Transit",
            r"NYC Ferry",
            r"Bee-Line",
            r"\bNICE\b",
            r"Suffolk County Transit",
        ],
    },
    "chicago": {
        "country_code": "US",
        "subdivision_names": ["Illinois", "Indiana"],
        "provider_patterns": [
            r"Chicago Transit Authority",
            r"\bCTA\b",
            r"\bMetra\b",
            r"\bPace\b",
            r"South Shore",
        ],
    },
    "washington_dc": {
        "country_code": "US",
        "subdivision_names": ["District of Columbia", "Maryland", "Virginia"],
        "provider_patterns": [
            r"\bWMATA\b",
            r"Washington Metropolitan",
            r"Ride On",
            r"Fairfax Connector",
            r"Arlington Transit",
            r"\bART\b",
            r"\bDASH\b",
            r"OmniRide",
            r"\bMARC\b",
            r"\bVRE\b",
            r"Virginia Railway Express",
        ],
    },
    "houston": {
        "country_code": "US",
        "subdivision_names": ["Texas"],
        "provider_patterns": [
            r"\bMETRO\b",
            r"Houston METRO",
            r"Fort Bend",
        ],
    },
    "atlanta": {
        "country_code": "US",
        "subdivision_names": ["Georgia"],
        "provider_patterns": [
            r"\bMARTA\b",
            r"CobbLinc",
            r"Ride Gwinnett",
            r"\bXpress\b",
            r"Douglas",
        ],
    },
    "phoenix": {
        "country_code": "US",
        "subdivision_names": ["Arizona"],
        "provider_patterns": [
            r"Valley Metro",
            r"Tempe Orbit",
            r"Scottsdale Trolley",
        ],
    },
}

print(f"Cities configured: {list(CITY_RULES.keys())}")
print(f"API keys configured: {len(API_KEYS)}")

Cities configured: ['new_york', 'chicago', 'washington_dc', 'houston', 'atlanta', 'phoenix']
API keys configured: 0


## 2. Catalog Helper Functions

In [3]:
def load_catalog(url: str = CATALOG_URL) -> pd.DataFrame:
    """Load and normalize the Mobility Database catalog."""
    df = pd.read_csv(url)

    # Flatten common nested columns from feeds_v2.csv if they exist
    rename_map = {
        "location.country_code": "country_code",
        "location.subdivision_name": "subdivision_name",
        "location.municipality": "municipality",
        "urls.latest": "latest_url",
        "urls.direct_download": "direct_download_url",
        "urls.authentication_type": "authentication_type",
        "urls.authentication_info_url": "authentication_info_url",
        "urls.api_key_parameter_name": "api_key_parameter_name",
    }
    for old, new in rename_map.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})

    # Normalize common fields
    for col in [
        "provider", "name", "country_code", "subdivision_name", "municipality",
        "status", "data_type", "authentication_type", "latest_url",
        "direct_download_url", "authentication_info_url", "api_key_parameter_name"
    ]:
        if col not in df.columns:
            df[col] = pd.NA

    return df


def normalize_official_flag(x) -> Optional[bool]:
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in {"true", "1", "yes"}:
        return True
    if s in {"false", "0", "no"}:
        return False
    return None


def clean_catalog(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and prepare catalog for matching."""
    df = df.copy()
    df["is_official_bool"] = df["is_official"].map(normalize_official_flag) if "is_official" in df.columns else None
    df["provider_name"] = (
        df["provider"].fillna("").astype(str) + " | " + df["name"].fillna("").astype(str)
    ).str.strip(" |")
    df["download_url"] = df["latest_url"].fillna(df["direct_download_url"])
    return df


def filter_base_gtfs(df: pd.DataFrame) -> pd.DataFrame:
    """Filter to active US GTFS feeds with download URLs."""
    out = df.copy()
    out = out[out["data_type"].astype(str).str.lower() == "gtfs"]
    out = out[out["country_code"].astype(str).str.upper() == "US"]
    out = out[out["status"].fillna("").isin(["active", "future", "inactive"])]
    out = out[out["download_url"].notna()]
    return out


def match_city_feeds(df: pd.DataFrame, city_rules: dict) -> pd.DataFrame:
    """Match feeds to a city using subdivision and provider patterns."""
    subds = city_rules["subdivision_names"]
    pats = city_rules["provider_patterns"]

    mask_subd = df["subdivision_name"].fillna("").isin(subds)
    mask_provider = pd.Series(False, index=df.index)

    provider_text = (
        df["provider"].fillna("").astype(str) + " " +
        df["name"].fillna("").astype(str) + " " +
        df["municipality"].fillna("").astype(str)
    )

    for pat in pats:
        mask_provider |= provider_text.str.contains(pat, case=False, regex=True, na=False)

    out = df[mask_subd & mask_provider].copy()

    # Prefer official feeds when duplicates exist
    out["official_rank"] = out["is_official_bool"].map({True: 2, None: 1, False: 0})
    out["status_rank"] = out["status"].map({"active": 3, "future": 2, "inactive": 1}).fillna(0)

    sort_cols = ["provider_name", "official_rank", "status_rank"]
    out = out.sort_values(sort_cols, ascending=[True, False, False])

    # De-duplicate by feed id if present, otherwise by provider + URL
    if "id" in out.columns:
        out = out.drop_duplicates(subset=["id"])
    else:
        out = out.drop_duplicates(subset=["provider_name", "download_url"])

    return out


print("Helper functions loaded.")

Helper functions loaded.


In [4]:
def resolve_auth_headers_and_params(row: pd.Series) -> tuple[dict, dict]:
    """Resolve authentication headers/params for a feed."""
    headers = {}
    params = {}

    auth_type = str(row.get("authentication_type", "")).strip()
    key_name = str(row.get("api_key_parameter_name", "")).strip()
    url = str(row.get("download_url", ""))

    if auth_type in {"1", "2"}:
        api_key = None

        # crude lookup by key name or URL substring
        for k, v in API_KEYS.items():
            if k.lower() in url.lower() or k.lower() == key_name.lower():
                api_key = v
                break

        if not api_key:
            raise RuntimeError(
                f"Feed requires authentication (type={auth_type}) but no API key was provided. "
                f"provider={row.get('provider')} info={row.get('authentication_info_url')}"
            )

        if auth_type == "1":
            params[key_name] = api_key
        elif auth_type == "2":
            headers[key_name] = api_key

    return headers, params


def safe_filename(text: str) -> str:
    """Convert text to safe filename."""
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text.strip())
    return text[:180] if len(text) > 180 else text


def download_feed(row: pd.Series, city_name: str, out_dir: Path, overwrite: bool = False) -> Path:
    """Download a single GTFS feed."""
    city_dir = out_dir / city_name
    city_dir.mkdir(parents=True, exist_ok=True)

    feed_id = str(row.get("id", "unknown_feed"))
    provider = str(row.get("provider", "unknown_provider"))
    filename = safe_filename(f"{provider}__{feed_id}.zip")
    out_fp = city_dir / filename

    if out_fp.exists() and not overwrite:
        return out_fp

    headers, params = resolve_auth_headers_and_params(row)

    with requests.get(
        row["download_url"],
        headers=headers,
        params=params,
        stream=True,
        timeout=120,
        allow_redirects=True,
    ) as r:
        r.raise_for_status()
        content_type = r.headers.get("Content-Type", "")
        if "zip" not in content_type.lower() and not str(row["download_url"]).lower().endswith(".zip"):
            print(f"  Warning: unexpected content-type for {provider}: {content_type}")

        with open(out_fp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return out_fp


def save_manifest(df: pd.DataFrame, city_name: str, out_dir: Path) -> None:
    """Save feed manifest for auditing."""
    city_dir = out_dir / city_name
    city_dir.mkdir(parents=True, exist_ok=True)
    manifest_fp = city_dir / "manifest.csv"
    df.to_csv(manifest_fp, index=False)


print("Download functions loaded.")

Download functions loaded.


In [5]:
# Load and filter catalog
print("Loading Mobility Database catalog...")
raw_catalog = load_catalog(CATALOG_URL)
catalog = clean_catalog(raw_catalog)
catalog = filter_base_gtfs(catalog)

print(f"Catalog rows after base filtering: {len(catalog):,}")
print(f"Columns: {list(catalog.columns)[:10]}...")

Loading Mobility Database catalog...
Catalog rows after base filtering: 1,262
Columns: ['id', 'data_type', 'entity_type', 'country_code', 'subdivision_name', 'municipality', 'provider', 'is_official', 'name', 'note']...


## 3. Download GTFS Feeds for All Cities

In [6]:
# Download GTFS feeds for all cities
download_summary = []

for city_name, rules in CITY_RULES.items():
    city_df = match_city_feeds(catalog, rules).copy()

    # Keep the most useful columns for auditing
    keep_cols = [
        c for c in [
            "id", "provider", "name", "municipality", "subdivision_name", "country_code",
            "status", "is_official", "download_url",
            "direct_download_url", "latest_url",
            "authentication_type", "authentication_info_url", "api_key_parameter_name"
        ] if c in city_df.columns
    ]
    city_df = city_df[keep_cols].sort_values(["provider", "name"], na_position="last")

    save_manifest(city_df, city_name, GTFS_DIR)

    print(f"\n{'='*60}")
    print(f"=== {city_name.upper()} ===")
    print(f"{'='*60}")
    print(city_df[["provider", "name", "municipality", "status"]].to_string(index=False))
    print(f"\nMatched feeds: {len(city_df)}")

    downloaded = 0
    skipped_existing = 0
    skipped_auth = 0
    failed = 0

    for _, row in city_df.iterrows():
        try:
            fp = download_feed(row, city_name, GTFS_DIR, overwrite=False)
            if fp.exists():
                if fp.stat().st_size > 0:
                    print(f"  ✓ {fp.name}")
                    downloaded += 1
                else:
                    skipped_existing += 1
            time.sleep(0.25)
        except RuntimeError as e:
            print(f"  ⚠ Skipped (auth): {row.get('provider')}")
            skipped_auth += 1
        except Exception as e:
            print(f"  ✗ Failed: {row.get('provider')} | {e}")
            failed += 1

    download_summary.append({
        "city": city_name,
        "matched_feeds": len(city_df),
        "downloaded": downloaded,
        "skipped_auth": skipped_auth,
        "failed": failed,
    })

# Save summary
summary_df = pd.DataFrame(download_summary)
summary_df.to_csv(GTFS_DIR / "download_summary.csv", index=False)

print("\n" + "="*60)
print("DOWNLOAD SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))


=== NEW_YORK ===
                                                                provider                    name    municipality   status
                                                MTA Metro-North Railroad                     NaN    Bronx County   active
                                    Metropolitan Transit Authority (MTA)               Bronx Bus    Bronx County   active
                                    Metropolitan Transit Authority (MTA)            Brooklyn Bus    Kings County   active
                                    Metropolitan Transit Authority (MTA)   Long Island Rail Road    Kings County   active
                                    Metropolitan Transit Authority (MTA)           Manhattan Bus New York County   active
                                    Metropolitan Transit Authority (MTA)         NYC Bus Company    Bronx County   active
                                    Metropolitan Transit Authority (MTA)              NYC Subway    Bronx County   active
      

## 4. Extract POI Locations for Routing

Create a file with unique POI locations (lat/lon) for each city to use as routing origins.

In [10]:
# Check what columns are available in the data
sample_file = list((CITIES_DIR / 'new_york').glob('part_*.parquet'))[0]
sample_df = pd.read_parquet(sample_file, columns=None)
print("Available columns:")
print(sample_df.columns.tolist())

Available columns:
['POI_CBG', 'SUB_CATEGORY', 'VISITOR_HOME_CBGS', 'ID_STORE', 'VISIT_COUNTS', 'DATE_RANGE_START', 'DATE_RANGE_END', 'LATITUDE', 'LONGITUDE', 'study_city', 'unified_category']


In [11]:
def extract_poi_locations(city_name: str, cities_dir: Path) -> pd.DataFrame:
    """Extract unique POI locations from city data."""
    
    city_dir = cities_dir / city_name
    part_files = sorted(city_dir.glob('part_*.parquet'))
    
    if not part_files:
        print(f"No data files found for {city_name}")
        return pd.DataFrame()
    
    # Read only location columns from all files
    # Check which columns exist
    sample = pd.read_parquet(part_files[0])
    
    # Try different possible column names
    lat_col = None
    lon_col = None
    id_col = None
    
    for col in sample.columns:
        col_lower = col.lower()
        if 'lat' in col_lower:
            lat_col = col
        if 'lon' in col_lower or 'lng' in col_lower:
            lon_col = col
        if col in ['ID_STORE', 'PLACEKEY', 'poi_id']:
            id_col = col
    
    if not lat_col or not lon_col:
        print(f"Could not find lat/lon columns. Available: {sample.columns.tolist()}")
        return pd.DataFrame()
    
    print(f"Using columns: id={id_col}, lat={lat_col}, lon={lon_col}")
    
    # Read all files
    cols_to_read = [c for c in [id_col, lat_col, lon_col] if c is not None]
    dfs = [pd.read_parquet(f, columns=cols_to_read) for f in part_files]
    df = pd.concat(dfs, ignore_index=True)
    
    # Get unique POIs
    if id_col:
        df_unique = df.drop_duplicates(subset=[id_col]).copy()
        df_unique = df_unique.rename(columns={id_col: 'poi_id', lat_col: 'lat', lon_col: 'lon'})
    else:
        df_unique = df.drop_duplicates(subset=[lat_col, lon_col]).copy()
        df_unique['poi_id'] = range(len(df_unique))
        df_unique = df_unique.rename(columns={lat_col: 'lat', lon_col: 'lon'})
    
    # Filter valid coordinates
    df_unique = df_unique[
        df_unique['lat'].notna() & 
        df_unique['lon'].notna() &
        (df_unique['lat'] != 0) &
        (df_unique['lon'] != 0)
    ]
    
    df_unique['city'] = city_name
    
    return df_unique[['poi_id', 'lat', 'lon', 'city']]

In [12]:
# Extract POI locations for all cities
all_pois = []

for city_name in CITY_RULES.keys():
    print(f"\nExtracting POIs for {city_name}...")
    df_pois = extract_poi_locations(city_name, CITIES_DIR)
    if len(df_pois) > 0:
        all_pois.append(df_pois)
        print(f"  Unique POIs: {len(df_pois):,}")

if all_pois:
    df_all_pois = pd.concat(all_pois, ignore_index=True)
    print(f"\nTotal POIs across all cities: {len(df_all_pois):,}")


Extracting POIs for new_york...
Using columns: id=ID_STORE, lat=LATITUDE, lon=LONGITUDE
  Unique POIs: 290,270

Extracting POIs for chicago...
Using columns: id=ID_STORE, lat=LATITUDE, lon=LONGITUDE
  Unique POIs: 140,675

Extracting POIs for washington_dc...
Using columns: id=ID_STORE, lat=LATITUDE, lon=LONGITUDE
  Unique POIs: 84,650

Extracting POIs for houston...
Using columns: id=ID_STORE, lat=LATITUDE, lon=LONGITUDE
  Unique POIs: 120,735

Extracting POIs for atlanta...
Using columns: id=ID_STORE, lat=LATITUDE, lon=LONGITUDE
  Unique POIs: 102,181

Extracting POIs for phoenix...
Using columns: id=ID_STORE, lat=LATITUDE, lon=LONGITUDE
  Unique POIs: 80,728

Total POIs across all cities: 819,239


In [13]:
# Save POI locations
if 'df_all_pois' in dir() and len(df_all_pois) > 0:
    # Save per city (for r5r routing)
    for city_name in df_all_pois['city'].unique():
        city_pois = df_all_pois[df_all_pois['city'] == city_name][['poi_id', 'lat', 'lon']]
        
        # r5r expects 'id', 'lat', 'lon' columns
        city_pois = city_pois.rename(columns={'poi_id': 'id'})
        
        output_file = ROUTING_DIR / f'{city_name}_poi_locations.csv'
        city_pois.to_csv(output_file, index=False)
        print(f"Saved: {output_file} ({len(city_pois):,} POIs)")
    
    # Save combined
    combined_file = ROUTING_DIR / 'all_poi_locations.parquet'
    df_all_pois.to_parquet(combined_file, index=False)
    print(f"\nSaved combined: {combined_file}")

Saved: dbs/routing/new_york_poi_locations.csv (290,270 POIs)
Saved: dbs/routing/chicago_poi_locations.csv (140,675 POIs)
Saved: dbs/routing/washington_dc_poi_locations.csv (84,650 POIs)
Saved: dbs/routing/houston_poi_locations.csv (120,735 POIs)
Saved: dbs/routing/atlanta_poi_locations.csv (102,181 POIs)
Saved: dbs/routing/phoenix_poi_locations.csv (80,728 POIs)

Saved combined: dbs/routing/all_poi_locations.parquet


## 5. Download Census Tract Centroids

For computing catchment diversity, we need to know which tracts are reachable from each POI. We'll use tract centroids as destinations.

In [14]:
# Load census tract data (from notebook 12)
# Updated to use correct filename from ACS 2024 download
census_file = Path('dbs/us_census/acs2024_tracts_study_cities.parquet')

if census_file.exists():
    df_tracts = pd.read_parquet(census_file)
    print(f"Loaded {len(df_tracts):,} tracts from census data")
    print(f"Columns: {df_tracts.columns.tolist()}")
else:
    print("Census data not found. Run notebook 12 first.")
    print(f"Expected: {census_file}")

Loaded 12,921 tracts from census data
Columns: ['NAME', 'median_household_income', 'total_population', 'nativity_total', 'native_born', 'foreign_born', 'foreign_born_total', 'foreign_europe', 'foreign_asia', 'foreign_africa', 'foreign_oceania', 'foreign_americas', 'race_total', 'race_white_alone', 'race_black_alone', 'race_asian_alone', 'race_hispanic_latino', 'state', 'county', 'tract', 'GEOID', 'county_fips', 'city', 'pct_foreign_born', 'pct_native_born', 'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic', 'pct_foreign_europe', 'pct_foreign_asia', 'pct_foreign_africa', 'pct_foreign_americas', 'income_quintile']


In [ ]:
# We need tract centroids for routing
# These can be obtained from Census TIGER/Line files
# For now, we'll use the Census Geocoder or download tract shapefiles

# Census tract centroids URL (TIGER/Line)
TIGER_TRACT_URL = "https://www2.census.gov/geo/tiger/TIGER2023/TRACT/"

print("""\n Download tract shapefiles and compute centroids
  - Download from: https://www.census.gov/cgi-bin/geo/shapefiles/index.php
  - Select Year: 2024, Layer Type: Census Tracts
  - Download for each state in study MSAs
  - Compute centroids with geopandas
""")

In [17]:
# State FIPS codes for our study cities
STATES_NEEDED = {
    '04': 'Arizona',
    '11': 'District of Columbia', 
    '13': 'Georgia',
    '17': 'Illinois',
    '18': 'Indiana',
    '24': 'Maryland',
    '34': 'New Jersey',
    '36': 'New York',
    '42': 'Pennsylvania',
    '48': 'Texas',
    '51': 'Virginia',
    '54': 'West Virginia',
    '55': 'Wisconsin',
}

print(f"States to download tract shapefiles: {len(STATES_NEEDED)}")
for fips, name in STATES_NEEDED.items():
    print(f"  {fips}: {name}")

States to download tract shapefiles: 13
  04: Arizona
  11: District of Columbia
  13: Georgia
  17: Illinois
  18: Indiana
  24: Maryland
  34: New Jersey
  36: New York
  42: Pennsylvania
  48: Texas
  51: Virginia
  54: West Virginia
  55: Wisconsin


In [18]:
def download_tract_shapefile(state_fips: str, output_dir: Path) -> Path:
    """Download Census tract shapefile for a state."""
    
    url = f"https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_{state_fips}_tract.zip"
    output_file = output_dir / f"tl_2024_{state_fips}_tract.zip"
    
    if output_file.exists():
        print(f"  Already exists: {output_file.name}")
        return output_file
    
    print(f"  Downloading {state_fips}...")
    response = requests.get(url, timeout=120)
    
    if response.status_code == 200:
        with open(output_file, 'wb') as f:
            f.write(response.content)
        print(f"  Saved: {output_file.name} ({len(response.content)/1e6:.1f} MB)")
        return output_file
    else:
        print(f"  ERROR: HTTP {response.status_code}")
        return None

# Download tract shapefiles
TRACT_DIR = Path('dbs/census_geo')
TRACT_DIR.mkdir(parents=True, exist_ok=True)

for state_fips in STATES_NEEDED.keys():
    download_tract_shapefile(state_fips, TRACT_DIR)

  Saved: tl_2024_04_tract.zip (8.5 MB)
  Saved: tl_2024_11_tract.zip (0.4 MB)
  Saved: tl_2024_13_tract.zip (16.0 MB)
  Saved: tl_2024_17_tract.zip (10.0 MB)
  Saved: tl_2024_18_tract.zip (6.3 MB)
  Saved: tl_2024_24_tract.zip (5.8 MB)
  Saved: tl_2024_34_tract.zip (7.1 MB)
  Saved: tl_2024_36_tract.zip (8.9 MB)
  Saved: tl_2024_42_tract.zip (13.1 MB)
  Saved: tl_2024_48_tract.zip (32.6 MB)
  Saved: tl_2024_51_tract.zip (16.8 MB)
  Saved: tl_2024_54_tract.zip (8.0 MB)
  Saved: tl_2024_55_tract.zip (7.6 MB)


In [ ]:
# Compute tract centroids using geopandas
try:
    import geopandas as gpd
    
    all_tracts_geo = []
    
    for state_fips in STATES_NEEDED.keys():
        shapefile = TRACT_DIR / f"tl_2024_{state_fips}_tract.zip"
        if shapefile.exists():
            print(f"Reading {state_fips}...")
            gdf = gpd.read_file(f"zip://{shapefile}")
            all_tracts_geo.append(gdf)
    
    if all_tracts_geo:
        gdf_all = pd.concat(all_tracts_geo, ignore_index=True)
        print(f"\nLoaded {len(gdf_all):,} tracts")
        
        # Compute centroids
        gdf_all['centroid'] = gdf_all.geometry.centroid
        gdf_all['centroid_lat'] = gdf_all.centroid.y
        gdf_all['centroid_lon'] = gdf_all.centroid.x
        
        # Create tract centroids table
        tract_centroids = gdf_all[['GEOID', 'centroid_lat', 'centroid_lon']].copy()
        tract_centroids = tract_centroids.rename(columns={
            'GEOID': 'tract_geoid',
            'centroid_lat': 'lat',
            'centroid_lon': 'lon'
        })
        
        print(f"Computed centroids for {len(tract_centroids):,} tracts")
        tract_centroids.head()
        
except ImportError:
    print("geopandas not installed. Install with: pip install geopandas")

In [20]:
# Save tract centroids
if 'tract_centroids' in dir():
    # Filter to study city tracts only
    if 'df_tracts' in dir():
        # Use GEOID column from census data (not tract_geoid)
        study_tract_ids = set(df_tracts['GEOID'].astype(str))
        tract_centroids_filtered = tract_centroids[
            tract_centroids['tract_geoid'].isin(study_tract_ids)
        ]
        print(f"Filtered to {len(tract_centroids_filtered):,} study city tracts")
    else:
        tract_centroids_filtered = tract_centroids
    
    # Save for r5r (expects 'id', 'lat', 'lon')
    tract_centroids_r5r = tract_centroids_filtered.rename(columns={'tract_geoid': 'id'})
    
    output_file = ROUTING_DIR / 'tract_centroids.csv'
    tract_centroids_r5r.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

Filtered to 12,921 study city tracts
Saved: dbs/routing/tract_centroids.csv


## 6. Create r5r Configuration

In [23]:
# r5r configuration for each city
r5r_config = {
    'parameters': {
        'max_walk_time': 15,       # minutes
        'max_trip_duration': 45,   # minutes
        'walk_speed': 4.5,         # km/h
        'departure_datetime': '2024-03-15 08:00:00',  # Typical weekday morning
        'time_window': 60,         # minutes (7:30–8:30 AM)
        'percentiles': [50],       # Median travel time
    },
    'cities': {}
}

for city_name in CITY_RULES.keys():
    
    city_gtfs_dir = GTFS_DIR / city_name
    city_output_dir = ROUTING_DIR / city_name

    city_config = {
        # Folder containing OSM.pbf + multiple GTFS zip files
        'transport_network_dir': str(city_gtfs_dir),

        # POI locations for accessibility analysis
        'poi_file': str(ROUTING_DIR / f'{city_name}_poi_locations.csv'),

        # Output directory
        'output_dir': str(city_output_dir),
    }

    r5r_config['cities'][city_name] = city_config
    
    # Create output directory
    city_output_dir.mkdir(parents=True, exist_ok=True)

# Save config
config_file = ROUTING_DIR / 'r5r_config.json'

with open(config_file, 'w') as f:
    json.dump(r5r_config, f, indent=2)

print(f"Saved: {config_file}")

print("\nConfiguration parameters:")
print(json.dumps(r5r_config['parameters'], indent=2))

Saved: dbs/routing/r5r_config.json

Configuration parameters:
{
  "max_walk_time": 15,
  "max_trip_duration": 45,
  "walk_speed": 4.5,
  "departure_datetime": "2024-03-15 08:00:00",
  "time_window": 60,
  "percentiles": [
    50
  ]
}


In [24]:
city_config

{'transport_network_dir': 'dbs/gtfs/phoenix',
 'poi_file': 'dbs/routing/phoenix_poi_locations.csv',
 'output_dir': 'dbs/routing/phoenix'}

## 7. Generate R Script Template

In [ ]:
r_script = '''
# Transit Catchment Diversity - r5r Routing
#
# This script computes transit travel times from POIs to all tract centroids
# within a given time threshold using r5r, and then aggregates the results
# so that each POI stores the set of reachable census tracts.
#
# Usage:
#   Rscript r_scripts/compute_transit_catchment.R --city new_york --max_time 30

library(r5r)
library(data.table)
library(jsonlite)
library(optparse)
library(arrow)

# Parse arguments
option_list <- list(
  make_option(c("-c", "--city"), type="character", default="new_york",
              help="City name [default: %default]"),
  make_option(c("-m", "--max_time"), type="integer", default=45,
              help="Maximum trip duration in minutes [default: %default]")
)
opt <- parse_args(OptionParser(option_list = option_list))

# Load configuration
config <- fromJSON("dbs/routing/r5r_config.json")
city_config <- config$cities[[opt$city]]

cat(sprintf("Processing %s...\\n", opt$city))

# Set Java memory before building network
options(java.parameters = "-Xmx8G")

# Build network from folder containing OSM + all GTFS zip files
r5r_core <- setup_r5(data_path = city_config$transport_network_dir)

# Load origins (POIs)
origins <- fread(city_config$poi_file)

# Load destinations (tract centroids)
destinations <- fread("dbs/routing/tract_centroids.csv")

# Basic checks
required_origin_cols <- c("id", "lat", "lon")
required_dest_cols <- c("id", "lat", "lon")

missing_origin_cols <- setdiff(required_origin_cols, names(origins))
missing_dest_cols <- setdiff(required_dest_cols, names(destinations))

if (length(missing_origin_cols) > 0) {
  stop(sprintf("Origins file is missing required columns: %s",
               paste(missing_origin_cols, collapse = ", ")))
}
if (length(missing_dest_cols) > 0) {
  stop(sprintf("Destinations file is missing required columns: %s",
               paste(missing_dest_cols, collapse = ", ")))
}

cat(sprintf("Origins (POIs): %d\\n", nrow(origins)))
cat(sprintf("Destinations (tracts): %d\\n", nrow(destinations)))

# Optionally filter destinations to the same city if a city column exists
if ("city" %in% names(destinations)) {
  destinations <- destinations[city == opt$city]
  cat(sprintf("Filtered destinations for %s: %d\\n", opt$city, nrow(destinations)))
}

# Compute travel time matrix
ttm <- travel_time_matrix(
  r5r_core,
  origins = origins,
  destinations = destinations,
  mode = c("WALK", "TRANSIT"),
  departure_datetime = as.POSIXct(config$parameters$departure_datetime),
  max_walk_time = config$parameters$max_walk_time,
  max_trip_duration = opt$max_time,
  time_window = config$parameters$time_window,
  percentiles = config$parameters$percentiles,
  progress = TRUE
)

cat(sprintf("Travel time matrix rows: %d\\n", nrow(ttm)))

# r5r usually returns one row per OD pair with a travel time column such as
# travel_time_p50 when percentiles = 50, or travel_time in some versions.
time_col <- NULL
candidate_time_cols <- c("travel_time_p50", "travel_time")
for (col in candidate_time_cols) {
  if (col %in% names(ttm)) {
    time_col <- col
    break
  }
}

if (is.null(time_col)) {
  stop("Could not find a travel time column in the r5r output.")
}

cat(sprintf("Using travel time column: %s\\n", time_col))

# Keep only OD pairs reachable within threshold and with non-missing travel time
ttm_reachable <- ttm[!is.na(get(time_col)) & get(time_col) <= opt$max_time]

cat(sprintf("Reachable OD pairs within %d min: %d\\n", opt$max_time, nrow(ttm_reachable)))

# Aggregate by POI:
# - number of reachable tracts
# - tract IDs stored as a single string
# - minimum / median travel time among reachable tracts for reference
catchment_summary <- ttm_reachable[
  ,
  .(
    reachable_tract_count = uniqueN(to_id),
    reachable_tract_ids = paste(sort(unique(to_id)), collapse = "|"),
    min_travel_time = min(get(time_col), na.rm = TRUE),
    median_travel_time = median(get(time_col), na.rm = TRUE)
  ),
  by = .(from_id)
]

# Rename from_id -> poi_id for clarity
setnames(catchment_summary, "from_id", "poi_id")

# Ensure POIs with zero reachable tracts are also kept
poi_base <- unique(origins[, .(poi_id = id)])
catchment_summary <- merge(
  poi_base,
  catchment_summary,
  by = "poi_id",
  all.x = TRUE
)

# Fill zeros / empty strings for POIs with no reachable tracts
catchment_summary[is.na(reachable_tract_count), reachable_tract_count := 0L]
catchment_summary[is.na(reachable_tract_ids), reachable_tract_ids := ""]
catchment_summary[is.na(min_travel_time), min_travel_time := NA_real_]
catchment_summary[is.na(median_travel_time), median_travel_time := NA_real_]

# Also attach POI coordinates if present
poi_info_cols <- intersect(c("id", "lat", "lon", "name", "category", "subcategory"), names(origins))
poi_info <- unique(origins[, ..poi_info_cols])
if ("id" %in% names(poi_info)) {
  setnames(poi_info, "id", "poi_id")
  catchment_summary <- merge(catchment_summary, poi_info, by = "poi_id", all.x = TRUE)
}

# Save aggregated POI-level catchment summary
output_file <- file.path(
  city_config$output_dir,
  sprintf("transit_catchment_summary_%dmin.parquet", opt$max_time)
)
write_parquet(catchment_summary, output_file)
cat(sprintf("Saved summary: %s\\n", output_file))

# Optionally also save the full reachable OD pairs
od_output_file <- file.path(
  city_config$output_dir,
  sprintf("transit_catchment_od_pairs_%dmin.parquet", opt$max_time)
)
write_parquet(ttm_reachable, od_output_file)
cat(sprintf("Saved reachable OD pairs: %s\\n", od_output_file))

# Cleanup
stop_r5(r5r_core)
'''

# Save R script
r_scripts_dir = Path('r_scripts')
r_scripts_dir.mkdir(parents=True, exist_ok=True)

r_script_file = r_scripts_dir / 'compute_transit_catchment.R'
with open(r_script_file, 'w') as f:
    f.write(r_script)

print(f"Saved: {r_script_file}")
print("\nTo run:")
print(f"  Rscript {r_script_file} --city new_york")

Saved: r_scripts/compute_transit_catchment.R

To run:
  Rscript r_scripts/compute_transit_catchment.R --city new_york


## 8. Summary

In [26]:
print("=" * 60)
print("TRANSIT CATCHMENT DATA PREPARATION SUMMARY")
print("=" * 60)

# Check what's ready
gtfs_ready = []
gtfs_merged = []
for city in CITY_RULES.keys():
    city_dir = GTFS_DIR / city
    if city_dir.exists():
        feeds = [f for f in city_dir.glob('*.zip') 
                 if f.name not in ('merged_gtfs.zip', 'manifest.csv')]
        if feeds:
            gtfs_ready.append(city)
        if (city_dir / 'merged_gtfs.zip').exists():
            gtfs_merged.append(city)

poi_ready = []
for city in CITY_RULES.keys():
    poi_file = ROUTING_DIR / f'{city}_poi_locations.csv'
    if poi_file.exists():
        poi_ready.append(city)

tract_centroids_ready = (ROUTING_DIR / 'tract_centroids.csv').exists()

print(f"""
DATA STATUS:
  GTFS feeds downloaded: {len(gtfs_ready)}/6 cities
    {', '.join(gtfs_ready) if gtfs_ready else 'None'}
  
  GTFS feeds merged: {len(gtfs_merged)}/6 cities
    {', '.join(gtfs_merged) if gtfs_merged else 'None'}
  
  POI locations extracted: {len(poi_ready)}/6 cities
    {', '.join(poi_ready) if poi_ready else 'None'}
  
  Tract centroids: {'✓ Ready' if tract_centroids_ready else '✗ Not ready'}

OUTPUT FILES:
  - dbs/gtfs/{{city}}/manifest.csv (feed audit trail)
  - dbs/gtfs/{{city}}/merged_gtfs.zip (combined GTFS per city)
  - dbs/routing/{{city}}_poi_locations.csv (POI coordinates)
  - dbs/routing/tract_centroids.csv (destination coordinates)
  - dbs/routing/r5r_config.json (routing parameters)
  - r_scripts/compute_transit_catchment.R (R script)

NEXT STEPS:
  1. Add API keys if feeds were skipped (check manifest.csv)
  2. Download OSM PBF files for each city (see instructions below)
  3. Install r5r in R: install.packages('r5r')
  4. Run: Rscript r_scripts/compute_transit_catchment.R --city {{city}}
""")

TRANSIT CATCHMENT DATA PREPARATION SUMMARY

DATA STATUS:
  GTFS feeds downloaded: 6/6 cities
    new_york, chicago, washington_dc, houston, atlanta, phoenix

  GTFS feeds merged: 0/6 cities
    None

  POI locations extracted: 6/6 cities
    new_york, chicago, washington_dc, houston, atlanta, phoenix

  Tract centroids: ✓ Ready

OUTPUT FILES:
  - dbs/gtfs/{city}/manifest.csv (feed audit trail)
  - dbs/gtfs/{city}/merged_gtfs.zip (combined GTFS per city)
  - dbs/routing/{city}_poi_locations.csv (POI coordinates)
  - dbs/routing/tract_centroids.csv (destination coordinates)
  - dbs/routing/r5r_config.json (routing parameters)
  - r_scripts/compute_transit_catchment.R (R script)

NEXT STEPS:
  1. Add API keys if feeds were skipped (check manifest.csv)
  2. Download OSM PBF files for each city (see instructions below)
  3. Install r5r in R: install.packages('r5r')
  4. Run: Rscript r_scripts/compute_transit_catchment.R --city {city}



## 9. OSM Data Download Instructions

In [27]:
osm_instructions = """
OSM PBF DOWNLOAD INSTRUCTIONS
=============================

r5r requires OpenStreetMap road network data in PBF format.

Download from Geofabrik:
  https://download.geofabrik.de/north-america/us.html

Files needed:
  - New York: new-york-latest.osm.pbf (or us-northeast)
  - Chicago: illinois-latest.osm.pbf
  - Washington DC: district-of-columbia-latest.osm.pbf + maryland + virginia
  - Houston: texas-latest.osm.pbf
  - Atlanta: georgia-latest.osm.pbf
  - Phoenix: arizona-latest.osm.pbf

Place files in:
  dbs/gtfs/{city_name}/

r5r will automatically detect .pbf files in the GTFS directory.

Alternative: Use osmextract R package to download automatically:
  library(osmextract)
  oe_get("New York", download_directory = "dbs/gtfs/new_york/")
"""

print(osm_instructions)


OSM PBF DOWNLOAD INSTRUCTIONS

r5r requires OpenStreetMap road network data in PBF format.

Download from Geofabrik:
  https://download.geofabrik.de/north-america/us.html

Files needed:
  - New York: new-york-latest.osm.pbf (or us-northeast)
  - Chicago: illinois-latest.osm.pbf
  - Washington DC: district-of-columbia-latest.osm.pbf + maryland + virginia
  - Houston: texas-latest.osm.pbf
  - Atlanta: georgia-latest.osm.pbf
  - Phoenix: arizona-latest.osm.pbf

Place files in:
  dbs/gtfs/{city_name}/

r5r will automatically detect .pbf files in the GTFS directory.

Alternative: Use osmextract R package to download automatically:
  library(osmextract)
  oe_get("New York", download_directory = "dbs/gtfs/new_york/")



## 10. Check the service date ranges

In [5]:
gtfs_dir = Path("dbs/gtfs/chicago")

def read_csv_from_zip(zf, name):
    with zf.open(name) as f:
        return pd.read_csv(f, dtype=str)

rows = []

for fp in gtfs_dir.glob("*.zip"):
    try:
        with zipfile.ZipFile(fp) as zf:
            has_cal = "calendar.txt" in zf.namelist()
            has_cal_dates = "calendar_dates.txt" in zf.namelist()

            start_min = end_max = None
            cal_dates_min = cal_dates_max = None

            if has_cal:
                cal = read_csv_from_zip(zf, "calendar.txt")
                if "start_date" in cal.columns:
                    start_min = cal["start_date"].min()
                if "end_date" in cal.columns:
                    end_max = cal["end_date"].max()

            if has_cal_dates:
                cd = read_csv_from_zip(zf, "calendar_dates.txt")
                if "date" in cd.columns and len(cd) > 0:
                    cal_dates_min = cd["date"].min()
                    cal_dates_max = cd["date"].max()

            rows.append({
                "feed": fp.name,
                "has_calendar": has_cal,
                "has_calendar_dates": has_cal_dates,
                "calendar_start_min": start_min,
                "calendar_end_max": end_max,
                "calendar_dates_min": cal_dates_min,
                "calendar_dates_max": cal_dates_max,
            })
    except Exception as e:
        rows.append({"feed": fp.name, "error": str(e)})

df = pd.DataFrame(rows)
print(df)

               feed  has_calendar  has_calendar_dates calendar_start_min  \
0  cleaned_gtfs.zip          True                True           20250101   

  calendar_end_max calendar_dates_min calendar_dates_max  
0         20271231           20250102           20261225  


## 11. Check the results

In [6]:
import pandas as pd
%cd /workspace

df_r = pd.read_parquet("dbs/routing/atlanta/catchment_summary_walk_transit_45min.parquet")
df_r.head()

/workspace


,poi_id,reachable_tract_count,reachable_tract_ids,min_travel_time,median_travel_time,lat,lon,city,max_time_min,routing_mode
0,10000047,2,13097080106|13097080107,24.0,27.0,33.793732,-84.661217,atlanta,45,WALK+TRANSIT
1,10000070,0,,NaN,NaN,33.566132,-84.195221,atlanta,45,WALK+TRANSIT
2,10000076,9,13121004100|13121004200|13121006000|1312100610...,24.0,41.0,33.729435,-84.460600,atlanta,45,WALK+TRANSIT
3,10000101,0,,NaN,NaN,33.707981,-84.889847,atlanta,45,WALK+TRANSIT
4,10000110,0,,NaN,NaN,33.870651,-84.830704,atlanta,45,WALK+TRANSIT
